In [3]:
import pandas as pd 
import numpy as np
import xgboost as xgb
import lightgbm as lgb
import catboost as cb
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.compose import ColumnTransformer
from sklearn.metrics import log_loss

In [5]:
results = pd.read_parquet('../data/interim/results_clean.parquet')
print(results.head())
print(results.shape)

        date home_team away_team  home_score  away_score tournament     city  \
0 1872-11-30  Scotland   England           0           0   Friendly  Glasgow   
1 1873-03-08   England  Scotland           4           2   Friendly   London   
2 1874-03-07  Scotland   England           2           1   Friendly  Glasgow   
3 1875-03-06   England  Scotland           2           2   Friendly   London   
4 1876-03-04  Scotland   England           3           0   Friendly  Glasgow   

    country  neutral  
0  Scotland    False  
1   England    False  
2  Scotland    False  
3   England    False  
4  Scotland    False  
(49482, 9)


In [10]:
# Same data scope + split as Model 1, so both models are evaluated on the identical eval set
model_df = results[results['date'].dt.year >= 1970].reset_index(drop=True)

# Leakage-safe split — boundaries mutually exclusive so no match is in two slices
wc2026_mask = (model_df['tournament'] == 'FIFA World Cup') & (model_df['date'].dt.year == 2026)

train_df = model_df[model_df['date'] < '2024-01-01']                   
eval_df = model_df[(model_df['date'] >= '2024-01-01') & ~wc2026_mask]  # held-out test set (WC excluded)
wc2026_df = model_df[wc2026_mask]                                      # forecast target, held aside

print(f"train {len(train_df)} | eval {len(eval_df)} | wc2026 {len(wc2026_df)}")

train 38900 | eval 2543 | wc2026 79
